# DBSCAN — Point Cloud Segmentation

Beginner-friendly notebook. Runs top to bottom.

DBSCAN unsupervised হলেও notebook-এর গঠন PointTransformerV2-এর সাথে হুবহু এক:
Config → Loader → Features → Metrics → MLflow → Algorithm → Parameter Grid →
Training (parameter search + early stopping + checkpoint/resume) → Visualization →
Inference helper → Testing → Final Inference।

PTv2-এর **epoch** = DBSCAN-এর **trial** (একটা eps + min_samples combination)।


In [1]:
# ── Install (run once if needed) ─────────────────────────────────────────────
# pip install laspy open3d numpy scikit-learn joblib tqdm mlflow matplotlib pandas

import os, glob, copy, random, logging
import numpy as np
import open3d as o3d
import laspy
import joblib
import mlflow
import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import tqdm
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# ── Configuration — change values here, nowhere else ────────────────────────
CONFIG = {
    "train_dir"      : "../data/train",   # labelled .las files
    "test_dir"       : "../data/test",    # unlabelled .las files for inference
    "checkpoint_dir" : "../checkpoints",  # where the best parameters are saved
    "num_classes"    : 2,                 # 0 = environment, 1 = wood powder
    "target_class"   : 1,                 # class we want to highlight (green)
    "val_ratio"      : 0.15,
    "test_ratio"     : 0.15,
    "seed"           : 42,
    # ── training (= parameter search) ──
    # PTv2-এর epoch loop-এর জায়গায় DBSCAN-এ (eps, min_samples) grid search চলে।
    # প্রতিটা combination = একটা trial. একই early stopping নীতি প্রযোজ্য।
    "patience"       : 8,                # এতগুলো trial-এ improvement না হলে থামবে
    "min_miou"       : 0.90,             # এই val mIoU পেলে সাথে সাথে থামবে
    # ── DBSCAN parameter grid ──
    "eps_values"         : [0.02, 0.03, 0.05, 0.08, 0.10, 0.15, 0.20],
    "min_samples_values" : [20, 30, 50, 100, 200, 300],
    # ── cluster selection strategy ──
    # "largest" | "highest" | "densest"  — কোন cluster-টা wood powder ধরা হবে
    "selection"      : "largest",
    # ── feature space ──
    # "xyz" (শুধু coordinates) | "xyz+feat" (xyz + height + density)
    "feature_space"  : "xyz+feat",
    # ── visualization ──
    "max_vis_files"  : 3,                # how many test files to show in 3D
    # ── MLflow ──
    "mlflow_uri"     : "http://localhost:5000",
    "mlflow_experiment": "DBSCAN_Segmentation",
}

os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
NUM_CLASSES = CONFIG["num_classes"]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

set_seed(CONFIG["seed"])


In [3]:
# ── Point cloud loader ───────────────────────────────────────────────────────
def load_pointcloud(path):
    """Read a .las/.laz file. Returns (points[N,3], labels[N] or None)."""
    las = laspy.read(path)
    pts = np.column_stack([np.asarray(las.x),
                           np.asarray(las.y),
                           np.asarray(las.z)]).astype(np.float64)
    labels = None
    for key in ("classification", "label", "labels", "class"):
        if key in las.point_format.dimension_names:
            labels = np.asarray(getattr(las, key), dtype=np.int64)
            break
    return pts, labels

def list_files(folder):
    """Return sorted list of .las/.laz files in a folder."""
    files = []
    for ext in (".las", ".laz"):
        files += glob.glob(os.path.join(folder, "*" + ext))
    return sorted(files)

# ── Split labelled files into train / val / test ─────────────────────────────
all_files = list_files(CONFIG["train_dir"])
assert all_files, f"No .las files found in {CONFIG['train_dir']}"

rng   = np.random.RandomState(CONFIG["seed"])
order = rng.permutation(len(all_files))
n_val  = max(1, int(len(all_files) * CONFIG["val_ratio"]))
n_test = max(1, int(len(all_files) * CONFIG["test_ratio"]))

VAL_FILES   = [all_files[i] for i in order[:n_val]]
TEST_FILES  = [all_files[i] for i in order[n_val : n_val + n_test]]
TRAIN_FILES = [all_files[i] for i in order[n_val + n_test :]]
INFER_FILES = list_files(CONFIG["test_dir"])   # no labels, final inference only

log.info(f"train={len(TRAIN_FILES)}  val={len(VAL_FILES)}  "
         f"test={len(TEST_FILES)}  inference={len(INFER_FILES)}")


2026-07-10 04:46:54,308 | INFO | train=33  val=6  test=6  inference=18


In [4]:
# ── Feature computation (DBSCAN-এর ইনপুট) ───────────────────────────────────
# PTv2 যেমন 7-channel feature ব্যবহার করত, DBSCAN এখানে 3 বা 5-channel ব্যবহার করে।
def compute_features(pts):
    """DBSCAN clustering-এর জন্য feature vector।

    'xyz'      → normalize করা XYZ (3 channels)
    'xyz+feat' → XYZ + height_above_floor + local_point_density (5 channels)
                 height: pile উঁচুতে থাকে, environment নিচে
                 density: pile-এর surface ঘন, দেয়াল/মেঝে আলাদা density-তে
    """
    pts_min  = pts.min(axis=0)
    pts_max  = pts.max(axis=0)
    norm_xyz = (pts - pts_min) / np.clip(pts_max - pts_min, 1e-9, None)

    if CONFIG["feature_space"] == "xyz":
        return norm_xyz.astype(np.float32)

    # height above floor
    z      = pts[:, 2]
    height = (z - z.min()) / max(z.max() - z.min(), 1e-9)

    # local point density (kNN mean distance-এর inverse)
    k    = 16
    nbrs = NearestNeighbors(n_neighbors=k + 1, algorithm="kd_tree",
                            n_jobs=-1).fit(pts)
    d, _ = nbrs.kneighbors(pts)
    mean_d  = d[:, 1:].mean(axis=1)              # skip self (distance 0)
    density = 1.0 / np.clip(mean_d, 1e-9, None)
    density = (density - density.min()) / max(density.max() - density.min(), 1e-9)

    return np.column_stack([norm_xyz, height, density]).astype(np.float32)


In [5]:
# ── Metrics ──────────────────────────────────────────────────────────────────
def compute_miou(true_labels, pred_labels, num_classes):
    """Compute mean Intersection-over-Union across all classes."""
    ious = []
    for c in range(num_classes):
        tp = int(((true_labels == c) & (pred_labels == c)).sum())
        fp = int(((true_labels != c) & (pred_labels == c)).sum())
        fn = int(((true_labels == c) & (pred_labels != c)).sum())
        if tp + fp + fn > 0:
            ious.append(tp / (tp + fp + fn))
    return float(np.mean(ious)) if ious else 0.0


def compute_dice(true_labels, pred_labels, num_classes):
    """Compute mean Dice Score across all classes."""
    dices = []
    for c in range(num_classes):
        tp = int(((true_labels == c) & (pred_labels == c)).sum())
        fp = int(((true_labels != c) & (pred_labels == c)).sum())
        fn = int(((true_labels == c) & (pred_labels != c)).sum())
        if tp + fp + fn > 0:
            # Dice Formula: 2*TP / (2*TP + FP + FN)
            dices.append((2 * tp) / (2 * tp + fp + fn))
    return float(np.mean(dices)) if dices else 0.0


def compute_accuracy(true_labels, pred_labels):
    """Point-wise classification accuracy."""
    return float((true_labels == pred_labels).mean())


def compute_val_loss(true_labels, pred_labels, num_classes):
    """DBSCAN-এ gradient/loss নেই — তাই (1 - mIoU) কে val_loss হিসেবে
    ব্যবহার করা হয়। mIoU বাড়লে এটা কমে, PTv2-এর val_loss-এর মতোই আচরণ।"""
    return 1.0 - compute_miou(true_labels, pred_labels, num_classes)


In [6]:
# ── MLflow setup ─────────────────────────────────────────────────────────────
try:
    mlflow.set_tracking_uri(CONFIG["mlflow_uri"])
    mlflow.set_experiment(CONFIG["mlflow_experiment"])
    MLFLOW_OK = True
    log.info(f"MLflow tracking: {CONFIG['mlflow_uri']}")
except Exception as e:
    MLFLOW_OK = False
    log.warning(f"MLflow not available ({e}) — training continues without logging")


2026-07-10 04:46:54,352 | WARNING | Retrying (Retry(total=6, connect=6, read=7, redirect=7, status=7)) after connection broken by 'NewConnectionError("HTTPConnection(host='localhost', port=5000): Failed to establish a new connection: [Errno 111] Connection refused")': /api/2.0/mlflow/experiments/get-by-name?experiment_name=DBSCAN_Segmentation
2026-07-10 04:46:58,992 | WARNING | Retrying (Retry(total=5, connect=5, read=7, redirect=7, status=7)) after connection broken by 'NewConnectionError("HTTPConnection(host='localhost', port=5000): Failed to establish a new connection: [Errno 111] Connection refused")': /api/2.0/mlflow/experiments/get-by-name?experiment_name=DBSCAN_Segmentation
2026-07-10 04:47:07,018 | WARNING | Retrying (Retry(total=4, connect=4, read=7, redirect=7, status=7)) after connection broken by 'NewConnectionError("HTTPConnection(host='localhost', port=5000): Failed to establish a new connection: [Errno 111] Connection refused")': /api/2.0/mlflow/experiments/get-by-name?e

## Algorithm

In [7]:
MODEL_NAME = "DBSCAN"

# DBSCAN-এ neural network নেই — "model" মানে segmentation algorithm নিজেই।
# দুটো ধাপ: (1) DBSCAN clustering  (2) target cluster নির্বাচন

def select_target_cluster(pts, cluster_labels, method):
    """DBSCAN-এর অনেক cluster-এর মধ্যে কোনটা wood powder সেটা নির্বাচন করে।

    method = 'largest' : সবচেয়ে বেশি point-এর cluster
    method = 'highest' : সবচেয়ে উঁচু (mean z) cluster
    method = 'densest' : bounding-box volume-এর তুলনায় সবচেয়ে ঘন cluster
    """
    unique = [l for l in np.unique(cluster_labels) if l >= 0]   # noise (-1) বাদ
    if not unique:
        return -1

    if method == "largest":
        sizes = {l: (cluster_labels == l).sum() for l in unique}
        return max(sizes, key=sizes.get)

    elif method == "highest":
        means = {l: pts[cluster_labels == l, 2].mean() for l in unique}
        return max(means, key=means.get)

    elif method == "densest":
        scores = {}
        for l in unique:
            sub = pts[cluster_labels == l]
            rng = sub.max(0) - sub.min(0)
            vol = max(rng[0] * rng[1] * rng[2], 1e-9)
            scores[l] = len(sub) / vol
        return max(scores, key=scores.get)

    return unique[0]


def dbscan_segment(pts, eps, min_samples):
    """Full DBSCAN segmentation: features → cluster → target selection → labels.

    Returns: preds[N] (0/1), cluster_labels[N], n_clusters
    """
    feat        = compute_features(pts)
    feat_scaled = StandardScaler().fit_transform(feat)

    db             = DBSCAN(eps=eps, min_samples=min_samples,
                            algorithm="ball_tree", n_jobs=-1)
    cluster_labels = db.fit_predict(feat_scaled)

    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)

    target_cluster = select_target_cluster(pts, cluster_labels,
                                           CONFIG["selection"])

    preds = np.zeros(len(pts), dtype=np.int64)
    if target_cluster >= 0:
        preds[cluster_labels == target_cluster] = CONFIG["target_class"]

    return preds, cluster_labels, n_clusters

log.info(f"{MODEL_NAME} algorithm ready — selection='{CONFIG['selection']}', "
         f"feature_space='{CONFIG['feature_space']}'")


2026-07-10 04:51:00,270 | INFO | DBSCAN algorithm ready — selection='largest', feature_space='xyz+feat'


## Parameter Grid  (PTv2-এর Dataset & DataLoader-এর জায়গায়)

DBSCAN-এ Dataset/DataLoader লাগে না — তার জায়গায় parameter grid।
PTv2 যেমন epoch ধরে ধরে train করত, DBSCAN এই grid-এর প্রতিটা combination ধরে ধরে
চেষ্টা করবে (প্রতিটা combination = একটা trial)।

In [8]:
# ── Parameter grid (training-এর "data") ─────────────────────────────────────
# সব (eps, min_samples) combination-এর তালিকা তৈরি — training loop এটার
# উপর দিয়ে iterate করবে, PTv2 যেমন DataLoader-এর উপর দিয়ে করত।
PARAM_GRID = [(eps, ms)
              for eps in CONFIG["eps_values"]
              for ms  in CONFIG["min_samples_values"]]

log.info(f"Parameter grid: {len(PARAM_GRID)} combinations "
         f"({len(CONFIG['eps_values'])} eps × "
         f"{len(CONFIG['min_samples_values'])} min_samples)")


2026-07-10 04:51:00,287 | INFO | Parameter grid: 42 combinations (7 eps × 6 min_samples)


## Training  (parameter search with early stopping + checkpoint/resume)

In [9]:
# ── Training loop (parameter search) with checkpointing ─────────────────────
# PTv2-এর train_model-এর হুবহু নীতি:
#   প্রতিটা trial-এ:     val set-এ mIoU/Dice/Acc/Loss হিসাব → MLflow-এ log
#   best হলে:            best parameters checkpoint-এ save (joblib)
#   early stopping:      patience trial-এ improvement না হলে বা min_miou পেলে থামে
#   resume policy:       checkpoint আগে থেকে থাকলে search পুরো skip

def train_model(model_name):
    """Search the parameter grid, save the best parameters, log to MLflow.
    Returns dict: {"eps": ..., "min_samples": ..., "best_miou": ...}"""

    ckpt_path = os.path.join(CONFIG["checkpoint_dir"], f"{model_name}_best.joblib")

    # ── resume: checkpoint থাকলে training পুরো skip ─────────────────────────
    if os.path.exists(ckpt_path):
        log.info(f"Checkpoint found at {ckpt_path} — loading, skipping search.")
        best = joblib.load(ckpt_path)
        log.info(f"Loaded: eps={best['eps']}  min_samples={best['min_samples']}  "
                 f"val mIoU={best['best_miou']:.4f}")
        return best

    # ── validation ডেটা প্রস্তুত (সব VAL_FILES একসাথে) ───────────────────────
    # প্রতিটা val ফাইলে আলাদাভাবে segment করে গড় mIoU নেওয়া হয়
    val_data = []
    for path in VAL_FILES:
        pts, lbl = load_pointcloud(path)
        if lbl is not None:
            lbl = np.clip(lbl, 0, NUM_CLASSES - 1)
            val_data.append((os.path.basename(path), pts, lbl))
    assert val_data, "Validation ফাইলে কোনো label নেই — search অসম্ভব।"

    best_miou  = -1.0
    best_eps   = None
    best_ms    = None
    no_improve = 0
    history    = []

    run_name = f"{model_name}_seg"
    with (mlflow.start_run(run_name=run_name) if MLFLOW_OK
          else open(os.devnull, "w")) as _:

        if MLFLOW_OK:
            mlflow.log_params({
                "model"          : model_name,
                "trials_total"   : len(PARAM_GRID),
                "patience"       : CONFIG["patience"],
                "min_miou_target": CONFIG["min_miou"],
                "selection"      : CONFIG["selection"],
                "feature_space"  : CONFIG["feature_space"],
            })

        stop = False
        for trial, (eps, ms) in enumerate(PARAM_GRID, start=1):

            # ── এই trial-এর parameter দিয়ে সব val ফাইল segment ──────────────
            mious, dices, accs, losses = [], [], [], []
            for fname, pts, lbl in val_data:
                try:
                    preds, _, _ = dbscan_segment(pts, eps=eps, min_samples=ms)
                except Exception as e:
                    log.warning(f"  {fname}: eps={eps} ms={ms} ERROR {e}")
                    continue
                mious.append(compute_miou(lbl, preds, NUM_CLASSES))
                dices.append(compute_dice(lbl, preds, NUM_CLASSES))
                accs.append(compute_accuracy(lbl, preds))
                losses.append(compute_val_loss(lbl, preds, NUM_CLASSES))

            if not mious:
                continue

            miou     = float(np.mean(mious))
            dice     = float(np.mean(dices))
            acc      = float(np.mean(accs))
            val_loss = float(np.mean(losses))
            history.append({"trial": trial, "eps": eps, "min_samples": ms,
                            "val_miou": miou, "val_loss": val_loss})

            # ── Logging to Console (PTv2 format) ─────────────────────────────
            log.info(
                f"Trial {trial:3d}/{len(PARAM_GRID)} | "
                f"eps={eps:.3f} min_samples={ms:3d} | "
                f"Val Loss: {val_loss:.4f} | Val Acc: {acc:.4f} | "
                f"mIoU: {miou:.4f} | Dice: {dice:.4f}"
            )

            # ── Logging to MLflow ─────────────────────────────────────────────
            if MLFLOW_OK:
                mlflow.log_metrics({
                    "val_loss": val_loss,
                    "val_acc" : acc,
                    "val_miou": miou,
                    "val_dice": dice,
                    "eps"         : eps,
                    "min_samples" : ms,
                }, step=trial)

            # ── update best ───────────────────────────────────────────────────
            if miou > best_miou + 1e-4:
                best_miou  = miou
                best_eps   = eps
                best_ms    = ms
                no_improve = 0
                best = {"eps": best_eps, "min_samples": best_ms,
                        "best_miou": best_miou,
                        "selection": CONFIG["selection"],
                        "feature_space": CONFIG["feature_space"],
                        "history": history}
                joblib.dump(best, ckpt_path)
                log.info(f"  ✓ New best mIoU {best_miou:.4f} → saved {ckpt_path}")
            else:
                no_improve += 1

            # ── early stopping ────────────────────────────────────────────────
            if miou >= CONFIG["min_miou"]:
                log.info(f"Early stop at trial {trial}: mIoU {miou:.4f} ≥ "
                         f"target {CONFIG['min_miou']}")
                stop = True
            if no_improve >= CONFIG["patience"]:
                log.info(f"Early stop at trial {trial} "
                         f"(no improvement for {CONFIG['patience']} trials)")
                stop = True
            if stop:
                break

        if MLFLOW_OK:
            mlflow.log_metric("best_val_miou", best_miou)

    best = joblib.load(ckpt_path)
    log.info(f"Best parameters: eps={best['eps']}  "
             f"min_samples={best['min_samples']}  (val mIoU={best['best_miou']:.4f})")
    return best


In [10]:
best_params = train_model(MODEL_NAME)

# best parameters CONFIG-এ বসানো — এরপর সব inference এগুলো ব্যবহার করবে
CONFIG["eps"]         = best_params["eps"]
CONFIG["min_samples"] = best_params["min_samples"]


2026-07-10 04:51:40,579 | INFO | Trial   1/42 | eps=0.020 min_samples= 20 | Val Loss: 0.7655 | Val Acc: 0.4685 | mIoU: 0.2345 | Dice: 0.3156
2026-07-10 04:51:40,580 | INFO |   ✓ New best mIoU 0.2345 → saved ../checkpoints/DBSCAN_best.joblib
2026-07-10 04:52:19,238 | INFO | Trial   2/42 | eps=0.020 min_samples= 30 | Val Loss: 0.7638 | Val Acc: 0.4724 | mIoU: 0.2362 | Dice: 0.3170
2026-07-10 04:52:19,239 | INFO |   ✓ New best mIoU 0.2362 → saved ../checkpoints/DBSCAN_best.joblib
2026-07-10 04:53:01,070 | INFO | Trial   3/42 | eps=0.020 min_samples= 50 | Val Loss: 0.7636 | Val Acc: 0.4728 | mIoU: 0.2364 | Dice: 0.3171
2026-07-10 04:53:01,070 | INFO |   ✓ New best mIoU 0.2364 → saved ../checkpoints/DBSCAN_best.joblib
2026-07-10 04:53:48,902 | INFO | Trial   4/42 | eps=0.020 min_samples=100 | Val Loss: 0.7636 | Val Acc: 0.4728 | mIoU: 0.2364 | Dice: 0.3171
2026-07-10 04:54:27,484 | INFO | Trial   5/42 | eps=0.020 min_samples=200 | Val Loss: 0.7636 | Val Acc: 0.4728 | mIoU: 0.2364 | Dice: 0.

## Visualization helper

In [11]:
# ── Visualization: Open3D ────────────────────────────────────────────────────
# Green = target (wood powder)   |   Red = others (environment)
def visualize_segmentation(points, predictions, title="Segmentation"):
    """Open an Open3D window showing the segmentation result."""
    colors = np.zeros((len(points), 3), dtype=np.float64)
    colors[predictions == CONFIG["target_class"]] = [0.0, 0.8, 0.0]   # green
    colors[predictions != CONFIG["target_class"]] = [0.8, 0.0, 0.0]   # red

    pcd = o3d.geometry.PointCloud()
    # centre the cloud so it appears at the origin
    center = points.mean(axis=0)
    pcd.points = o3d.utility.Vector3dVector(
        (points - center).astype(np.float64))
    pcd.colors = o3d.utility.Vector3dVector(colors)

    # XYZ axes
    span = float((points.max(0) - points.min(0)).max()) * 0.5
    axes = o3d.geometry.TriangleMesh.create_coordinate_frame(size=span)

    n_target = int((predictions == CONFIG["target_class"]).sum())
    n_other  = len(predictions) - n_target
    full_title = (f"{title}  |  green(target)={n_target:,}  "
                  f"red(others)={n_other:,}")
    o3d.visualization.draw_geometries([pcd, axes],
                                       window_name=full_title,
                                       width=1280, height=800)


## Inference helper

In [12]:
# ── Full-cloud inference ──────────────────────────────────────────────────────
def predict_full_cloud(path):
    """Label every point in a file using the best DBSCAN parameters.
    PTv2-এর predict_full_cloud-এর মতোই signature: returns (pts, preds, lbl)."""
    pts, lbl = load_pointcloud(path)
    if lbl is not None:
        lbl = np.clip(lbl, 0, NUM_CLASSES - 1)
    preds, _, _ = dbscan_segment(pts,
                                 eps=CONFIG["eps"],
                                 min_samples=CONFIG["min_samples"])
    return pts, preds, lbl


## Volume Estimation (TIN)

Segmentation-এর পরে target points থেকে volume বের করা হয়।

**Pipeline (সব `vol_TIN()`-এর ভেতরে):**
1. **SOR** — noisy বিচ্ছিন্ন point বাদ (statistical distance filter)
2. **DBSCAN largest cluster** — stray mislabeled points বাদ
3. **TIN integration** — 2D Delaunay → Σ Area₂D(triangle) × mean height
   - Floor baseline = scanned z-এর `floor_pct` percentile (outer-shell scan-এ আলাদা floor point থাকে না)
   - কোনো alpha clipping নেই — DBSCAN-ই stray point সামলায়; clipping করলে কিনারার বৈধ triangle কেটে গিয়ে volume কমে যেত
4. **Bootstrap CI** — ৮০ বার subsample করে std + 95% confidence interval


In [ ]:
# ── Volume Estimation: TIN-based volumetric integration ─────────────────────
from scipy.spatial import Delaunay, cKDTree


def sor_filter(pts, k=16, std_ratio=2.0):
    """Statistical Outlier Removal: mean-kNN-distance > mean + std_ratio*std
    হলে সেই point বাদ। Returns (filtered_pts, kept_mask)."""
    pts = np.asarray(pts, np.float64)
    if len(pts) <= k + 1:
        return pts, np.ones(len(pts), bool)
    tree = cKDTree(pts)
    d, _ = tree.query(pts, k=k + 1)          # col-0 = নিজেই (দূরত্ব 0)
    mean_dist = d[:, 1:].mean(axis=1)
    thr  = mean_dist.mean() + std_ratio * mean_dist.std()
    mask = mean_dist <= thr
    return pts[mask], mask


def extract_main_cluster(pts, eps=0.15, min_samples=8):
    """DBSCAN দিয়ে সবচেয়ে বড় connected cluster রাখা — stray points বাদ।
    এর পরে আর alpha clipping লাগে না।"""
    pts = np.asarray(pts, np.float64)
    if len(pts) < min_samples * 2:
        return pts
    try:
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(pts)
        labels = np.asarray(pcd.cluster_dbscan(eps=eps, min_points=min_samples))
        valid = labels[labels >= 0]
        if valid.size == 0:
            return pts
        main_lbl = np.bincount(valid).argmax()
        result = pts[labels == main_lbl]
        return result if len(result) >= min_samples else pts
    except Exception as e:
        log.warning(f"[TIN] cluster extraction failed ({e}) -> using all points")
        return pts


def tin_integrate(pts, floor_pct=0.5):
    """Delaunay TIN integration.
    floor baseline = scanned z-এর floor_pct percentile (pile-এর base প্রান্ত)।
    Volume = Σ Area2D(triangle) × mean_height(triangle)."""
    pts = np.asarray(pts, np.float64)
    if len(pts) < 4:
        return float("nan")

    floor_z = np.percentile(pts[:, 2], floor_pct)
    h  = pts[:, 2] - floor_z
    xy = pts[:, :2]
    try:
        tri = Delaunay(xy)
    except Exception as e:
        log.warning(f"[TIN] Delaunay failed ({e})")
        return float("nan")

    t = tri.simplices
    p0, p1, p2 = xy[t[:, 0]], xy[t[:, 1]], xy[t[:, 2]]
    h0, h1, h2 = h[t[:, 0]], h[t[:, 1]], h[t[:, 2]]

    cross  = ((p1[:, 0] - p0[:, 0]) * (p2[:, 1] - p0[:, 1])
             - (p1[:, 1] - p0[:, 1]) * (p2[:, 0] - p0[:, 0]))
    area2d = np.abs(cross) / 2.0
    mean_h = (h0 + h1 + h2) / 3.0
    valid  = mean_h > 0
    return float(np.sum(area2d[valid] * mean_h[valid]))


def bootstrap_tin_confidence(pts, n_bootstrap=80, subsample_frac=0.85,
                             floor_pct=0.5, seed=0):
    """Bootstrap: বারবার subsample করে TIN চালিয়ে uncertainty মাপা।
    Returns {'std', 'ci_low', 'ci_high'} বা None।"""
    rng = np.random.RandomState(seed)
    n = len(pts)
    n_sub = max(10, int(n * subsample_frac))
    estimates = []
    for _ in range(n_bootstrap):
        idx = rng.choice(n, n_sub, replace=False)
        v = tin_integrate(pts[idx], floor_pct=floor_pct)
        if np.isfinite(v) and v > 0:
            estimates.append(v)
    if len(estimates) < 5:
        return None
    arr = np.array(estimates)
    return {"std":     round(float(arr.std()), 5),
            "ci_low":  round(float(np.percentile(arr, 2.5)), 5),
            "ci_high": round(float(np.percentile(arr, 97.5)), 5)}


def vol_TIN(pts, sor_k=16, sor_std=2.0, cluster_eps=0.15,
            floor_pct=0.5, run_bootstrap=True, n_bootstrap=80):
    """সম্পূর্ণ volume pipeline: SOR → largest cluster → TIN [→ bootstrap CI].
    Returns (volume, confidence_dict_or_None). একক = pts-এর এককের ঘন (m³)।"""
    pts = np.asarray(pts, np.float64)
    if len(pts) < 10:
        return float("nan"), None

    pts, _ = sor_filter(pts, k=sor_k, std_ratio=sor_std)          # 1. SOR
    if len(pts) < 10:
        return float("nan"), None

    pts = extract_main_cluster(pts, eps=cluster_eps)               # 2. cluster
    if len(pts) < 10:
        return float("nan"), None

    volume = tin_integrate(pts, floor_pct=floor_pct)               # 3. TIN

    conf = None                                                    # 4. CI
    if run_bootstrap and len(pts) >= 20:
        conf = bootstrap_tin_confidence(pts, n_bootstrap=n_bootstrap,
                                        floor_pct=floor_pct)
    return float(volume), conf


## Testing  (held-out test files with labels)

In [ ]:
# ── Testing on held-out test files ───────────────────────────────────────────
log.info("=== TESTING ===")
test_mious = []
for path in TEST_FILES:
    name = os.path.splitext(os.path.basename(path))[0]
    pts, preds, lbl = predict_full_cloud(path)
    miou = compute_miou(lbl, preds, NUM_CLASSES)
    test_mious.append(miou)
    log.info(f"  {name}: mIoU = {miou:.4f}")

log.info(f"Mean test mIoU: {np.mean(test_mious):.4f}")

# ── Visualize + volume for the first few test files ──────────────────────────
for path in TEST_FILES[:CONFIG["max_vis_files"]]:
    name = os.path.splitext(os.path.basename(path))[0]
    pts, preds, _ = predict_full_cloud(path)

    # segmentation → vol_TIN → volume
    target_pts   = pts[preds == CONFIG["target_class"]]
    volume, conf = vol_TIN(target_pts)

    n_tot = len(pts)
    n_tgt = len(target_pts)
    print(f"\n{'─'*55}")
    print(f"  File          : {name}")
    print(f"  Total Points  : {n_tot:,}")
    print(f"  Target Points : {n_tgt:,}")
    print(f"  Other Points  : {n_tot - n_tgt:,}")
    print(f"  Target Ratio  : {n_tgt / max(n_tot, 1) * 100:.2f}%")
    print(f"  Est. Volume   : {volume:.4f} m³")
    if conf:
        print(f"  95% CI        : [{conf['ci_low']:.4f} – {conf['ci_high']:.4f}] m³"
              f"  (±{conf['std']:.4f})")
    print(f"{'─'*55}")

    visualize_segmentation(pts, preds,
                           title=f"{MODEL_NAME} | {name} | V={volume:.4f} m³")


2026-07-10 04:59:12,402 | INFO | === TESTING ===
2026-07-10 04:59:17,213 | INFO |   sample_data_0013: mIoU = 0.1500
2026-07-10 04:59:23,822 | INFO |   sample_data_0020: mIoU = 0.1598
2026-07-10 04:59:34,118 | INFO |   sample_data_0017: mIoU = 0.1353
2026-07-10 04:59:38,745 | INFO |   sample_data_0012: mIoU = 0.1947
2026-07-10 04:59:43,242 | INFO |   sample_data_0015: mIoU = 0.1434
2026-07-10 04:59:46,527 | INFO |   sample_data_0031: mIoU = 0.0563
2026-07-10 04:59:46,527 | INFO | Mean test mIoU: 0.1399


## Final Inference  (`data/test`, no labels)

In [ ]:
# ── Final inference on data/test (no labels needed) ──────────────────────────
if INFER_FILES:
    log.info("=== FINAL INFERENCE ===")
    for path in INFER_FILES:
        name = os.path.splitext(os.path.basename(path))[0]
        pts, preds, _ = predict_full_cloud(path)

        # segmentation → vol_TIN → volume
        target_pts   = pts[preds == CONFIG["target_class"]]
        volume, conf = vol_TIN(target_pts)

        n_tot = len(pts)
        n_tgt = len(target_pts)
        print(f"\n{'─'*55}")
        print(f"  File          : {name}")
        print(f"  Total Points  : {n_tot:,}")
        print(f"  Target Points : {n_tgt:,}")
        print(f"  Other Points  : {n_tot - n_tgt:,}")
        print(f"  Target Ratio  : {n_tgt / max(n_tot, 1) * 100:.2f}%")
        print(f"  Est. Volume   : {volume:.4f} m³")
        if conf:
            print(f"  95% CI        : [{conf['ci_low']:.4f} – {conf['ci_high']:.4f}] m³"
                  f"  (±{conf['std']:.4f})")
        print(f"{'─'*55}")

        visualize_segmentation(pts, preds,
                               title=f"INFERENCE | {MODEL_NAME} | {name} | "
                                     f"V={volume:.4f} m³")
else:
    log.info("data/test is empty — skipping inference.")
